In [ ]:
import matplotlib
# matplotlib.use('Agg')
import matplotlib.pyplot as plt
import copy
import math
import numpy as np
from torchvision import datasets, transforms
import torch
from numpy import *

from utils.sampling import mnist_iid_cluster, mnist_noniid_cluster, cifar_iid
from utils.options import args_parser
from models_v4.Update import LocalUpdate,ClusterDetect, ProxUpdate
from models_v4.Nets import MLP, CNNMnist, CNNCifar, LeNet, CNNMnist2
from models_v4.Fed import FedAvg_vectorization
from models_v4.Fed import FedQAvg, FedBrea, Quantization, Quantization_Finite, my_score, my_score_Finite
from models_v4.test import test_img
from scipy.linalg import null_space


%load_ext autoreload
%autoreload 2
class my_argument:    
    epochs = 200   #"rounds of training"
    num_users = 5 # "number of users: K"
    frac = 0.1 #"the fraction of clients: C"
    local_ep=5 #"the number of local epochs: E"
    local_bs=50 #"local batch size: B"
    bs=128 #"test batch size"
    lr=0.0001 #"learning rate"
    momentum=0.5 # "SGD momentum (default: 0.5)"
    split='user' # "train-test split type, user or sample"

    # model arguments
    model = 'cnn'
    kernel_num=9 #, help='number of each kind of kernel')
    kernel_sizes='3,4,5' #  help='comma-separated kernel size to use for convolution')
    norm='batch_norm' #, help="batch_norm, layer_norm, or None")
    num_filters=32 #, help="number of filters for conv nets")
    max_pool='True' #help="Whether use max pooling rather than strided convolutions")

    # other arguments
    dataset='mnist' #, help="name of dataset")
    iid=0
    num_classes=10#, help="number of classes")
    num_channels=1#, help="number of channels of images")
    gpu=2#, help="GPU ID, -1 for CPU")
    stopping_rounds=10#, help='rounds of early stopping')
    verbose='False'#, help='verbose print')
    seed=1#, help='random seed (default: 1)')
    cluster=5
    opt='ADAM'
    mu=0.1
args = my_argument()
#args.device = torch.device('cuda:{}'.format(args.gpu) if torch.cuda.is_available() and args.gpu != -1 else 'cpu')
print(args.epochs)
np.random.seed(1)


In [ ]:
# load dataset and split users
if args.dataset == 'mnist':
    trans_mnist = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
    dataset_train = datasets.MNIST('./data/mnist/', train=True, download=True, transform=trans_mnist)
    dataset_test = datasets.MNIST('./data/mnist/', train=False, download=True, transform=trans_mnist)
    count=0
    #print(len(dataset_train))
    dict_users=[] #2D array in each row, users of a particular cluster
    train_data=[]
    test_data=[]
    for j in range(args.cluster):
        train_data.append([])
        test_data.append([])
        dict_users.append([])
    for j in range(len(dataset_train)):
        if (dataset_train.train_labels[j].numpy()==0) | (dataset_train.train_labels[j].numpy()==1):
            train_data[0].append(dataset_train[j])
        elif (dataset_train.train_labels[j].numpy()==2) | (dataset_train.train_labels[j].numpy()==3):
            train_data[1].append(dataset_train[j])
        elif (dataset_train.train_labels[j].numpy()==4) | (dataset_train.train_labels[j].numpy()==5):
            train_data[2].append(dataset_train[j])
        elif (dataset_train.train_labels[j].numpy()==6) | (dataset_train.train_labels[j].numpy()==7):
            train_data[3].append(dataset_train[j])
        elif (dataset_train.train_labels[j].numpy()==8) | (dataset_train.train_labels[j].numpy()==9):
            train_data[4].append(dataset_train[j])
    for j in range(len(dataset_test)):
        if (dataset_test.test_labels[j].numpy()==0) | (dataset_test.test_labels[j].numpy()==1):
            test_data[0].append(dataset_test[j])
        elif (dataset_test.test_labels[j].numpy()==2) | (dataset_test.test_labels[j].numpy()==3):
            test_data[1].append(dataset_test[j])
        elif (dataset_test.test_labels[j].numpy()==4) | (dataset_test.test_labels[j].numpy()==5):
            test_data[2].append(dataset_test[j])
        elif (dataset_test.test_labels[j].numpy()==6) | (dataset_test.test_labels[j].numpy()==7):
            test_data[3].append(dataset_test[j])
        elif (dataset_test.test_labels[j].numpy()==8) | (dataset_test.test_labels[j].numpy()==9):
            test_data[4].append(dataset_test[j])
    
#defining 5 different types of datasets for 5 different clusters
    
    if args.iid:
        for cluster_no in range(args.cluster):
            dict_users[cluster_no] = mnist_iid_cluster(train_data[cluster_no], args.num_users)
    else:
        for cluster_no in range(args.cluster):
            dict_users[cluster_no] = mnist_noniid_cluster(train_data[cluster_no], args.num_users)
elif args.dataset == 'cifar':
    trans_cifar = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
    dataset_train = datasets.CIFAR10('./data/cifar', train=True, download=True, transform=trans_cifar)
    dataset_test = datasets.CIFAR10('./data/cifar', train=False, download=True, transform=trans_cifar)
#defining 5 different types of datasets for 5 different clusters
    for j in range(len(dataset_train)):
        if (dataset_train.train_labels[j].numpy()==0) | (dataset_train.train_labels[j].numpy()==1):
            train_data[0].append(dataset_train[j])
        elif (dataset_train.train_labels[j].numpy()==2) | (dataset_train.train_labels[j].numpy()==3):
            train_data[1].append(dataset_train[j])
        elif (dataset_train.train_labels[j].numpy()==4) | (dataset_train.train_labels[j].numpy()==5):
            train_data[2].append(dataset_train[j])
        elif (dataset_train.train_labels[j].numpy()==6) | (dataset_train.train_labels[j].numpy()==7):
            train_data[3].append(dataset_train[j])
        elif (dataset_train.train_labels[j].numpy()==8) | (dataset_train.train_labels[j].numpy()==9):
            train_data[4].append(dataset_train[j])
    for j in range(len(dataset_test)):
        if (dataset_test.test_labels[j].numpy()==0) | (dataset_test.test_labels[j].numpy()==1):
            test_data[0].append(dataset_test[j])
        elif (dataset_test.test_labels[j].numpy()==2) | (dataset_test.test_labels[j].numpy()==3):
            test_data[1].append(dataset_test[j])
        elif (dataset_test.test_labels[j].numpy()==4) | (dataset_test.test_labels[j].numpy()==5):
            test_data[2].append(dataset_test[j])
        elif (dataset_test.test_labels[j].numpy()==6) | (dataset_test.test_labels[j].numpy()==7):
            test_data[3].append(dataset_test[j])
        elif (dataset_test.test_labels[j].numpy()==8) | (dataset_test.test_labels[j].numpy()==9):
            test_data[4].append(dataset_test[j])

    if args.iid:
         for cluster_no in range(args.cluster):
            dict_users[cluster_no] = cifar_cluster(train_data[cluster_no], args.num_users)
    else:
        exit('Error: only consider IID setting in CIFAR10')
else:
    exit('Error: unrecognized dataset')
img_size = dataset_train[0][0].shape
print((dict_users[0]))
print((dict_users[0][4]))
print(len(dataset_train))
print(len(dict_users))
print(len(train_data[0]))
#print(train_data[0])
print(len(dataset_train[0]))
#idxs_users = np.random.choice(range(args.num_users), m, replace=False)
#print(idxs_users)

In [ ]:
#use_cuda = torch.cuda.is_available()
#print(use_cuda)
#args.device = torch.device("cuda" if use_cuda else "cpu")
args.device=torch.device("cuda:3")
print(args.device)

In [ ]:
acc_test=[]
acc_test_arr=[]
loss_test=[]
loss_test_arr=[]
cluster_0_acc=[]
cluster_1_acc=[]
cluster_2_acc=[]
cluster_3_acc=[]
cluster_4_acc=[]

In [ ]:
import copy

In [ ]:
# build model
import numpy as np
import copy
from models_v4.Fed import FedAvg
if args.model == 'cnn' and args.dataset == 'cifar':
    net_glob=(CNNCifar(args=args).to(args.device))
elif args.model == 'cnn' and args.dataset == 'mnist':
    net_glob=(CNNMnist(args=args).to(args.device))
elif args.model == 'mlp':
    len_in = 1
    for x in img_size:
        len_in *= x
    net_glob=(MLP(dim_in=len_in, dim_hidden=200, dim_out=args.num_classes).to(args.device))
else:
    exit('Error: unrecognized model')
#print(net_glob)

# copy weights
from models_v4.Fed import weight_vectorization_gen
w_glob=net_glob.state_dict()
abs_vect,layer_size=weight_vectorization_gen(w_glob)
print(len(abs_vect))
#print("weights")
#print(w_glob)
#print(w_glob.shape)
#print(w_glob[2])

# training
cv_loss, cv_acc = [], []
val_loss_pre, counter = 0, 0
net_best = None
best_loss = None
val_acc_list, net_list = [], []

#hist_ = np.zeros(10,dtype=int)
sample=0 # fro the purpose of using fresh samples in each iteration
for iter in range(200): #args.epochs
    print("iteration number",iter)
    w_locals, loss_locals = [], []
    loss_train=[]
    cluster_block=[]
    idx_users=[]
    sorted_train_data=[]
    for cluster_no in range(args.cluster):
        for index in dict_users[cluster_no]:
            idx_users.append(index) # putting the data indices of users in this list
            sorted_train_data.append(train_data[cluster_no])#putting the corresponding training data in this array
    updated=[]
    model_diff=[]
    grad_vect=[]
    prev=[]
    error=[]
    grad_vect_send=[]
    store_grad=[]
    for i in range(len(idx_users)):
        updated.append([])
        model_diff.append([])
        grad_vect.append([])
        prev.append([])
        #error.append(np.zeros((d,1)))
        grad_vect_send.append([])
        store_grad.append([]) 
   
    for user2 in range(len(idx_users)):
            #local = LocalUpdate(args=args, dataset=sorted_train_data[user], idxs=idx_users[user][(sample+1)*600:(sample+2)*600])
            #local = LocalUpdate(args=args, dataset=sorted_train_data[user], idxs=idx_users[user][600:1199])
        updated[user2]=copy.deepcopy(w_glob)
        local = ProxUpdate(args=args, dataset=sorted_train_data[user2], idxs=idx_users[user2])
            #using 2nd half data
        w, loss = local.train3(net=copy.deepcopy(net_glob).to(args.device))
        w_locals.append(copy.deepcopy(w))
        loss_locals.append(copy.deepcopy(loss))
    w_glob = FedAvg(w_locals)
    # copy weight to net_glob
    net_glob.load_state_dict(w_glob)
     # print loss
    loss_avg = sum(loss_locals) / len(loss_locals)
    
    loss_train.append(loss_avg)
    
    acc_test, loss_test = test_img(net_glob, dataset_test, args)
    acc_test_arr.append(acc_test)
    loss_test_arr.append(loss_test)
    if iter % 1 ==0:
        print('Round {:3d}, Average loss {:.3f} Test accuracy {:.3f}'.format(iter, loss_avg,acc_test))
        #print(hist_)
    #print(loss_train)
    acc_test,loss_test=test_img(net_glob,test_data[0],args)
    cluster_0_acc.append(acc_test)
    acc_test,loss_test=test_img(net_glob,test_data[1],args)
    cluster_1_acc.append(acc_test)
    acc_test,loss_test=test_img(net_glob,test_data[2],args)
    cluster_2_acc.append(acc_test)
    acc_test,loss_test=test_img(net_glob,test_data[3],args)
    cluster_3_acc.append(acc_test)
    acc_test,loss_test=test_img(net_glob,test_data[4],args)
    cluster_4_acc.append(acc_test)
   
    
           
       

In [ ]:

plt.plot(range(len(cluster_0_acc)), cluster_0_acc)
plt.ylabel('cluser 0 test accuracy (%)')
plt.show()
plt.plot(range(len(cluster_1_acc)), cluster_1_acc)
plt.ylabel('cluser 1 test accuracy (%)')
plt.show()
plt.plot(range(len(cluster_2_acc)), cluster_2_acc)
plt.ylabel('cluser 2 test accuracy (%)')
plt.show()
plt.plot(range(len(cluster_3_acc)), cluster_3_acc)
plt.ylabel('cluser 3 test accuracy (%)')
plt.show()
plt.plot(range(len(cluster_4_acc)), cluster_4_acc)
plt.ylabel('cluser 4 test accuracy (%)')
plt.show()

In [ ]:
#print(acc_test_arr[39])
print(cluster_0_acc[199])
print(cluster_1_acc[199])
print(cluster_2_acc[199])
print(cluster_3_acc[199])
print(cluster_4_acc[199])

In [ ]:
print(cluster_0_acc)

In [ ]:
cluster_0=[]
cluster_1=[]
cluster_2=[]
cluster_3=[]
cluster_4=[]
for i in cluster_0_acc:
    cluster_0.append(float(i))
cluster_1=[]
for i in cluster_1_acc:
    cluster_1.append(float(i))
cluster_2=[]
for i in cluster_2_acc:
    cluster_2.append(float(i))
cluster_3=[]
for i in cluster_3_acc:
    cluster_3.append(float(i))
cluster_4=[]
for i in cluster_4_acc:
    cluster_4.append(float(i))
print(cluster_0)
print(cluster_1)
print(cluster_2)
print(cluster_3)
print(cluster_4)

In [ ]:
print(cluster_0)

In [ ]:
print(cluster_1)

In [ ]:
print(cluster_2)

In [ ]:
print(cluster_3)

In [ ]:
print(cluster_4)